<a href="https://colab.research.google.com/github/mritunjay-chandra-jha/operation_research_problems/blob/main/Transportation_Problems.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MEDSE 592 – Deterministic Decision Modeling Lab 7
### Transportation Problems: Problem Set (modelled and solved with Gurobi / `gurobipy`)

**Name:** **Mritunjay Chandra Jha**  **Roll No.:** **26222006**



**How this notebook is organised**
- Every model is a **function** (`build_p1`, `build_p1_balanced`, `build_p2`), so each what-if re-uses the same model with changed data.
- Each what-if starts again from the **original** model, never from the previous what-if.
- Sub-question layout: *code cell → output → Markdown answer*, labelled (a), (b), …
- Run with **Runtime → Run all**. Run the install cell first (again after any runtime restart).
- Gurobi sign convention used throughout: for a *minimisation* problem a `<=` constraint has a dual `Pi <= 0`, so the **value of one extra unit of capacity = −Pi**; a `>=` constraint has `Pi >= 0`; an `=` constraint has a free-sign dual.

In [ ]:
!pip install gurobipy -q

In [ ]:
import gurobipy as gp
from gurobipy import GRB

print("Gurobi Version", gp.gurobi.version())

Gurobi Version (13, 0, 3)


In [ ]:
# source (plants) and destination (demands) definitions
source = ["Nagpur", "Raipur", "Jabalpur"]
destination = ["Pune", "Hyderabad", "Bhopal", "Lucknow"]

supply = {"Nagpur": 350, "Raipur": 450, "Jabalpur": 300}
demand = {"Pune": 250, "Hyderabad": 300, "Bhopal": 200, "Lucknow": 280}

cost = {("Nagpur", "Pune"): 420, ("Nagpur", "Hyderabad"): 380, ("Nagpur","Bhopal"):300,   ("Nagpur","Lucknow"):650,
        ("Raipur", "Pune"): 560, ("Raipur","Hyderabad"):450,   ("Raipur","Bhopal"):420,   ("Raipur","Lucknow"):550,
        ("Jabalpur","Pune"):600, ("Jabalpur","Hyderabad"):520, ("Jabalpur","Bhopal"):240,  ("Jabalpur","Lucknow"):480}

print("Total supply:", sum(supply.values()), "| Total demand:", sum(demand.values()))

Total supply: 1100 | Total demand: 1030


Cement Distribution Model and Variables

In [ ]:
model = gp.Model("cement_distribution")
model.Params.OutputFlag = 0

# Variables represent shipments from source (plant) to destination (market)
x = model.addVars(source, destination, name="x", lb=0)

Restricted license - for non-production use only - expires 2027-11-29


Objective Function

In [ ]:
# Cost matrix is keyed as (source, destination)
model.setObjective(gp.quicksum(cost[s, d] * x[s, d] for s in source for d in destination), GRB.MINIMIZE)

Constraints

In [ ]:
# Supply constraints for each plant (source)
supply_constraint = model.addConstrs((gp.quicksum(x[s, d] for d in destination) <= supply[s] for s in source), name="supply")

# Demand constraints for each market (destination)
demand_constraint = model.addConstrs((gp.quicksum(x[s, d] for s in source) == demand[d] for d in destination), name="demand")

In [ ]:
model.optimize()

if model.status == GRB.OPTIMAL:
    print("Minimum total cost :", model.objVal)
    for s in source:
      for d in destination:
        if x[s, d].X > 1e-6:
          print(f" {s} -> {d}: {x[s, d].X:.0f} tonnes")
        else:
          print(f" {s} -> {d}: 0 tonnes")
else:
  print("No optimal solution. Status code:", model.Status)

Minimum total cost : 428000.0
 Nagpur -> Pune: 250 tonnes
 Nagpur -> Hyderabad: 100 tonnes
 Nagpur -> Bhopal: 0 tonnes
 Nagpur -> Lucknow: 0 tonnes
 Raipur -> Pune: 0 tonnes
 Raipur -> Hyderabad: 200 tonnes
 Raipur -> Bhopal: 0 tonnes
 Raipur -> Lucknow: 180 tonnes
 Jabalpur -> Pune: 0 tonnes
 Jabalpur -> Hyderabad: 0 tonnes
 Jabalpur -> Bhopal: 200 tonnes
 Jabalpur -> Lucknow: 100 tonnes


## **Problem 1 : (Solution)**

### (a) Balance check and LP formulation

**Totals**
- Total capacity = 350 + 450 + 300 = **1100 t**
- Total demand = 250 + 300 + 200 + 280 = **1030 t**

Capacity exceeds demand by 70 t, so the problem is **not balanced**. It has a supply surplus, which is feasible: some capacity can be left unused.

**Sets**
- I = {Nagpur, Raipur, Jabalpur} (plants)
- J = {Pune, Hyderabad, Bhopal, Lucknow} (cities)

**Parameters**
- $c_{ij}$ = cost of shipping one tonne from plant i to city j (₹/t)
- $s_i$ = weekly capacity of plant i (t)
- $d_j$ = weekly demand of city j (t)

**Decision variables**
- $x_{ij} \ge 0$ = tonnes shipped from plant i to city j

**Objective**

$$\min \; Z = \sum_{i \in I} \sum_{j \in J} c_{ij}\, x_{ij}$$

**Constraints**

$$\sum_{j \in J} x_{ij} \le s_i \quad \forall i \in I \qquad \text{(supply)}$$

$$\sum_{i \in I} x_{ij} = d_j \quad \forall j \in J \qquad \text{(demand)}$$

$$x_{ij} \ge 0 \quad \forall i \in I,\; j \in J$$

**Why these directions**
- **Supply is ≤.** A plant cannot ship more than it can produce, but it may produce less. Since total capacity (1100) is greater than total demand (1030), at least 70 t of capacity must stay unused. An "=" constraint would force every plant to run at full capacity and make the model infeasible.

- **Demand is =.** Each city's demand must be met in full, so shipping less is not allowed. Shipping more is allowed in principle but pointless, since every $c_{ij} > 0$ and it would only add cost. So "=" and "≥" give the same optimum, and "=" states the requirement most directly.



### (b) Solution

- **Solver status:** OPTIMAL (status code 2)
- **Minimum total cost:** ₹428,000 per week

**Shipment plan (tonnes)**

| Plant \ City | Pune | Hyderabad | Bhopal | Lucknow | Shipped | Capacity | Unused |
|---|---|---|---|---|---|---|---|
| Nagpur | 250 | 100 | 0 | 0 | 350 | 350 | **0** |
| Raipur | 0 | 200 | 0 | 180 | 380 | 450 | **70** |
| Jabalpur | 0 | 0 | 200 | 100 | 300 | 300 | **0** |

**Cost check:** Nagpur 250·420 + 100·380 = 143,000; Raipur 200·450 + 180·550 = 189,000; Jabalpur 200·240 + 100·480 = 96,000. Total = ₹428,000.

**Unused capacity:** Nagpur and Jabalpur run at full capacity. Raipur leaves **70 t** unused, which equals the surplus of total capacity (1100 t) over total demand (1030 t). Raipur is left with the spare capacity because it is the costliest plant to use on the margin.

In [ ]:
print("Reduced cost of unused routes:")
for s in source:
    for d in destination:
        if x[s, d].X < 1e-6:
            print(f"  {s} -> {d}: {x[s, d].RC:.0f}")

print("\nDual values:")
for s in source:      print(f"  supply {s}: {supply_constraint[s].Pi:.0f}")
for d in destination: print(f"  demand {d}: {demand_constraint[d].Pi:.0f}")

Reduced cost of unused routes:
  Nagpur -> Bhopal: 60
  Nagpur -> Lucknow: 170
  Raipur -> Pune: 70
  Raipur -> Bhopal: 110
  Jabalpur -> Pune: 180
  Jabalpur -> Hyderabad: 140

Dual values:
  supply Nagpur: -70
  supply Raipur: 0
  supply Jabalpur: -70
  demand Pune: 490
  demand Hyderabad: 450
  demand Bhopal: 310
  demand Lucknow: 550


### (c) Interpretation of Dual Values

Based on the optimization results, here are the economic interpretations in rupees:

#### **(i) Why is Raipur's supply dual value equal to zero?**
* **Economic Meaning:** Raipur has **70 tonnes of unused capacity** (it is a non-binding constraint).
* **Reasoning:** In optimization, if a resource is not fully utilized, having one more unit of it does not change the objective function. Having 451 tonnes of capacity instead of 450 does not lower costs because we already have surplus capacity that we are choosing not to use. Hence, its shadow price (marginal value) is **₹0**.

#### **(ii) Which city is the most expensive to serve at the margin?**
* **Economic Meaning:** The dual value of a demand constraint represents the cost of serving one additional tonne of demand to that city.
* **Reasoning:** Look at the demand dual values:
  * **Pune:** ₹420
  * **Hyderabad:** ₹380
  * **Bhopal:** ₹300
  * **Lucknow:** ₹550
* **Answer:** **Lucknow** is the most expensive city to serve at the margin, with a dual value of **₹550/t**. Increasing demand in Lucknow by 1 unit increases the minimum total cost by ₹550.

#### **(iii) What does a negative dual on a supply constraint mean?**
* **Economic Meaning:** Gurobi reports dual values based on the direction of optimization. For a minimization problem with `Ax <= b` constraints, a negative dual value (if encountered) means that **increasing** the supply capacity **decreases** the overall cost.
* **Reasoning:** Under supply constraints ($x_{ Nagpur, d} + x_{ Nagpur, d} \le 350$), the dual represents $\frac{\partial Z}{\partial \text{Supply}}$. If Gurobi reports Nagpur and Jabalpur with negative duals (e.g., -₹70), it means that adding 1 tonne of capacity at these cheaper plants allows us to shift production away from the more expensive Raipur plant, saving us ₹70 per unit.